# Preparación de Datos para la Integración de DataFrames

## 🎯 Objetivos
Al finalizar este notebook, serás capaz de:
- Cargar conjuntos de datos masivos de forma eficiente.
- Realizar una selección estratégica de columnas para optimizar la memoria.
- Identificar y validar "claves de unión" (Join Keys) entre diferentes fuentes de datos.
- Preparar DataFrames para operaciones de `merge` y `concat`.

## 📖 Introducción

En entornos reales, la información rara vez reside en una sola tabla. Los datos suelen estar fragmentados en múltiples archivos CSV, bases de datos o APIs. Por ejemplo, la información descriptiva de una película (título, género) suele estar en una tabla, mientras que sus métricas de éxito (votos, calificación) están en otra.

Antes de intentar unir estas tablas, es crítico realizar un proceso de **limpieza y selección**. Cargar millones de filas innecesarias puede agotar la memoria RAM (especialmente en hardware limitado), por lo que la regla de oro es: *Carga solo lo que necesites*.

In [ ]:
import pandas as pd
from pathlib import Path

# Configuración de rutas
DATA_PATH = Path(".")

## 1. Carga Eficiente de Datos

### 💡 Intuición
Cuando trabajamos con archivos CSV muy grandes, Pandas puede tener dificultades para inferir los tipos de datos de cada columna, lo que puede generar advertencias de `DtypeWarning`. El parámetro `low_memory=False` le indica a Pandas que procese el archivo de manera más robusta, aunque consuma un poco más de memoria inicialmente.

In [ ]:
# Carga de los datasets
# Nota: Usamos low_memory=False para evitar advertencias de tipos de datos en archivos grandes
df_movies = pd.read_csv(DATA_PATH / "IMDb movies.csv", low_memory=False)
df_ratings = pd.read_csv(DATA_PATH / "IMDb ratings.csv")

print(f"Dimensiones Movies: {df_movies.shape}")
print(f"Dimensiones Ratings: {df_ratings.shape}")

## 2. Selección Estratégica de Columnas

### 💡 Intuición
Imagina que tienes un libro de 1000 páginas pero solo necesitas 3 datos específicos de cada página. No llevarías el libro entero a la mesa de trabajo; solo anotarías esos 3 datos. En Pandas, hacemos lo mismo seleccionando solo las columnas esenciales para nuestro análisis.

### 🛠️ Diagrama de Preparación
```
 [ Dataset Movies Crudo ]       [ Dataset Ratings Crudo ]
         |                              |
         v                              v
 [ Seleccionar Columnas ]       [ Seleccionar Columnas ]
         |                              |
         v                              v
 [ Movies DF Optimizado ] <---Key---> [ Ratings DF Optimizado ]
```

In [ ]:
# Definimos las columnas que realmente nos interesan
movies_cols = ['imdb_title_id', 'title', 'year', 'genre', 'country']
ratings_cols = ['imdb_title_id', 'total_votes', 'mean_vote']

# Creamos nuevas versiones optimizadas de los DataFrames
df_movies_clean = df_movies[movies_cols].copy()
df_ratings_clean = df_ratings[ratings_cols].copy()

print("--- Vista previa Movies (Optimizado) ---")
display(df_movies_clean.head())

print("\n--- Vista previa Ratings (Optimizado) ---")
display(df_ratings_clean.head())

## 3. Identificación de la "Clave de Unión" (Join Key)

### 💡 Intuición
Para que dos tablas se puedan unir, deben compartir una "columna puente". Esta columna debe contener identificadores únicos que existan en ambas tablas. En este caso, la clave es `imdb_title_id`.

Si intentamos unir tablas sin una clave común o con claves que tienen formatos diferentes (ej. una es texto y la otra es número), la operación fallará o producirá resultados vacíos.

In [ ]:
# Verificamos que la clave sea la misma en ambos DataFrames
print(f"Columna clave en Movies: { 'imdb_title_id' in df_movies_clean.columns }")
print(f"Columna clave en Ratings: { 'imdb_title_id' in df_ratings_clean.columns }")

# Mostramos los primeros valores de la clave para validar el formato
print("\nEjemplo de claves en Movies:")
print(df_movies_clean['imdb_title_id'].head())
print("\nEjemplo de claves en Ratings:")
print(df_ratings_clean['imdb_title_id'].head())

## 📝 Ejercicios de Práctica

**Ejercicio 1**: Carga los datasets originales y calcula cuánta memoria RAM consumen antes y después de la selección de columnas utilizando `df.info()` o `df.memory_usage(deep=True).sum()`.

**Ejercicio 2**: Imagina que necesitas añadir la columna `director` al dataset de películas. Modifica la selección de columnas de `df_movies` para incluirla y verifica que la clave `imdb_title_id` siga presente.

In [ ]:
# Solución Ejercicio 1
# TODO: Implementar aquí
pass

In [ ]:
# Solución Ejercicio 2
# TODO: Implementar aquí
pass

## 📋 Resumen Rápido

1. **`low_memory=False`**: Evita errores de tipos de datos en CSVs grandes.
2. **Selección de Columnas**: `df[['col1', 'col2']]` reduce el consumo de RAM drásticamente.
3. **Join Key**: Identificador único compartido entre tablas necesario para cualquier operación de `merge` o `join`.